# LITTLE THINGS provenance check and dwarf ω recompute
**EPS Research, October 2026.** Independent verification run on Node1.

Purpose:
1. Test whether HI Corpus v7.1 (Zenodo 10.5281/zenodo.21768240) LITTLE THINGS records contain Oh et al. (2015) *Model* rows mixed in with the *Data* rows.
2. Identify the second curve against the source catalogue, VizieR J/AJ/149/180.
3. Recompute per-galaxy dwarf ω from the source `Data` rows only and compare with `omega_results_dwarfs_v1.1.csv`.

This notebook reads only Zenodo deposits and VizieR, and shares no code with the v1.1 script.

In [1]:
import json, hashlib, collections, subprocess, os
import numpy as np, pandas as pd
from astroquery.vizier import Vizier

FILES = {
    'v71.json':   'https://zenodo.org/records/21768240/files/rotation_curve_corpus_v7.json?download=1',
    'dwarf.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json?download=1',
}
for f, url in FILES.items():
    if not os.path.exists(f):
        subprocess.run(['curl', '-sSL', '-o', f, url], check=True)
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest())

v71.json a9d668ea9649be5473f161157ab20436177ba98abfa2c9fdb9ad1069bee81327
dwarf.json 452fba4108d29a21e5246b5446a7910bfcf085dc2371f770ec9db5702a944429


## 1. Does v7.1 contain an appended second curve?

In [2]:
G  = json.load(open('v71.json'))['galaxies']
v7 = {g['galaxy']: g for g in G if 'LITTLE' in str(g.get('survey', '')).upper()}

def split(d):
    for i in range(1, len(d)):
        if d[i]['Rad'] < d[i-1]['Rad']:
            return d[:i], d[i:]
    return d, []

rows = []
for n, g in v7.items():
    a, b = split(g['data'])
    rows.append(dict(galaxy=n, n_first=len(a), n_second=len(b)))
s1 = pd.DataFrame(rows)
print('LT galaxies:', len(v7), ' LT rows:', sum(len(g['data']) for g in v7.values()),
      ' with second block:', (s1.n_second > 0).sum(), ' second-block rows:', s1.n_second.sum())
s1

LT galaxies: 26  LT rows: 1716  with second block: 25  second-block rows: 839


,galaxy,n_first,n_second
0,CVnIdwA,14,16
1,DDO_210,8,8
2,DDO_216,13,13
3,DDO_53,14,13
4,DDO_70,24,24
5,IC_10,25,25
6,IC_1613,109,109
7,UGC_8508,25,25
8,WLM,140,126
9,DDO_50,85,85


## 2. What is the second curve? Comparison with Oh et al. (2015)

In [3]:
Vizier.ROW_LIMIT = -1
T = Vizier.get_catalogs('J/AJ/149/180/rotdmbar')[0]
gal = Vizier.get_catalogs('J/AJ/149/180/galaxies')[0]
print('rotdmbar rows:', len(T), ' Type counts:', collections.Counter(T['Type']))

def viz(name, typ):
    s = T[(T['Name'] == name) & (T['Type'] == typ)]
    R = np.array(s['R'] * s['R0.3'], float); V = np.array(s['V'] * s['V0.3'], float)
    E = np.array(s['e_V'] * s['V0.3'], float); o = np.argsort(R)
    return R[o], V[o], E[o]

# A: VizieR Data rows == v7.1 first block, row for row
chk = []
for n, g in v7.items():
    R, V, _ = viz(n, 'Data'); a, _ = split(g['data'])
    r7 = np.array([p['Rad'] for p in a]); vv = np.array([p['Vobs'] for p in a])
    chk.append(dict(galaxy=n, n_viz_data=len(R), n_v7_first=len(r7),
                    match=len(R) == len(r7) and np.allclose(R, r7, atol=0.006) and np.allclose(V, vv, atol=0.006)))
chk = pd.DataFrame(chk); print('Data rows match v7.1 first block:', chk.match.sum(), '/', len(chk))

# B: identify the Model curve via catalogue V(Rmax) and Viso(Rmax), DDO 154
r = gal[gal['Name'] == 'DDO_154'][0]
a, b = split(v7['DDO_154']['data'])
at = lambda blk, R0: min(blk, key=lambda p: abs(p['Rad'] - R0))['Vobs']
print(f"DDO_154 Rmax={r['Rmax']} kpc: catalogue V(Rmax)={r['V(Rmax)']}, Viso(Rmax)={r['Viso(Rmax)']}")
print(f"   v7.1 first block at Rmax: {at(a, r['Rmax'])}   second block at Rmax: {at(b, r['Rmax'])}")
chk

rotdmbar rows: 1716  Type counts: Counter({'Data': 877, 'Model': 839})
Data rows match v7.1 first block: 26 / 26
DDO_154 Rmax=7.320000171661377 kpc: catalogue V(Rmax)=47.79999923706055, Viso(Rmax)=49.20000076293945
   v7.1 first block at Rmax: 47.86   second block at Rmax: 49.2259


,galaxy,n_viz_data,n_v7_first,match
0,CVnIdwA,14,14,True
1,DDO_210,8,8,True
2,DDO_216,13,13,True
3,DDO_53,14,14,True
4,DDO_70,24,24,True
5,IC_10,25,25,True
6,IC_1613,109,109,True
7,UGC_8508,25,25,True
8,WLM,140,140,True
9,DDO_50,85,85,True


## 3. Independent dwarf ω from source `Data` rows (Eq. 6, named-term form)

In [4]:
dw = [g for g in json.load(open('dwarf.json'))['galaxies'] if g.get('omega_ready')]
out = []
for g in sorted(dw, key=lambda g: g['galaxy']):
    R, V, E = viz(g['galaxy'], 'Data')
    R1, V1, R2, V2 = R[0], V[0], R[-1], V[-1]
    outer_term = (V2 / R2)
    inner_term = (V1 / R1) * ((R1 / R2) ** 1.5)
    omega_kms_kpc = outer_term - inner_term          # Flynn & Cannaliato 2025 Eq.6  [km/s/kpc]
    omega_rad_gyr = omega_kms_kpc * 1.0227           # 1 km/s/kpc = 1.0227 rad/Gyr
    out.append(dict(galaxy=g['galaxy'], n_data=len(R), dup_R=len(R) - len(np.unique(np.round(R, 2))),
                    R1=R1, V1=V1, R2=R2, V2=V2,
                    omega_kms_kpc=omega_kms_kpc, omega_rad_gyr=omega_rad_gyr))
s3 = pd.DataFrame(out)
print('N =', len(s3), ' all positive:', bool((s3.omega_kms_kpc > 0).all()),
      ' repeated radii in Data rows:', int(s3.dup_R.sum()))
s3.round(4)

N = 24  all positive: True  repeated radii in Data rows: 0


,galaxy,n_data,dup_R,R1,V1,R2,V2,omega_kms_kpc,omega_rad_gyr
0,CVnIdwA,14,0,0.19,3.97,2.59,23.58,8.6891,8.8863
1,DDO_101,9,0,0.26,30.59,2.32,64.92,23.5687,24.1037
2,DDO_126,30,0,0.13,3.21,3.99,37.22,9.1831,9.3916
3,DDO_133,19,0,0.18,5.17,3.48,46.12,12.9150,13.2082
4,DDO_154,55,0,0.14,6.20,7.89,50.51,6.2971,6.4400
5,DDO_168,33,0,0.12,3.33,3.99,47.26,11.6999,11.9655
6,DDO_210,8,0,0.04,2.68,0.31,12.02,35.6688,36.4784
7,DDO_216,13,0,0.09,3.23,1.12,18.91,16.0664,16.4311
8,DDO_43,9,0,0.47,8.56,4.19,33.22,7.2442,7.4086
9,DDO_46,23,0,0.15,14.63,3.54,40.92,10.7086,10.9517


## 4. Comparison with `omega_results_dwarfs_v1.1.csv`
Place the v1.1 CSV in this folder. It is read only here, for the comparison.

In [5]:
v11 = pd.read_csv('omega_results_dwarfs_v1.1.csv')
m = s3.merge(v11[['galaxy', 'omega_kms_kpc', 'omega_rad_gyr']], on='galaxy', suffixes=('_indep', '_v11'))
m['abs_diff_kms_kpc'] = (m.omega_kms_kpc_indep - m.omega_kms_kpc_v11).abs()
print('galaxies compared:', len(m), '  max |difference| km/s/kpc:', m.abs_diff_kms_kpc.max().round(5))
m.round(4)

galaxies compared: 24   max |difference| km/s/kpc: 5e-05


,galaxy,n_data,dup_R,R1,V1,R2,V2,omega_kms_kpc_indep,omega_rad_gyr_indep,omega_kms_kpc_v11,omega_rad_gyr_v11,abs_diff_kms_kpc
0,CVnIdwA,14,0,0.19,3.97,2.59,23.58,8.6891,8.8863,8.6891,8.8863,0.0000
1,DDO_101,9,0,0.26,30.59,2.32,64.92,23.5687,24.1037,23.5687,24.1038,0.0000
2,DDO_126,30,0,0.13,3.21,3.99,37.22,9.1831,9.3916,9.1831,9.3916,0.0000
3,DDO_133,19,0,0.18,5.17,3.48,46.12,12.9150,13.2082,12.9150,13.2082,0.0000
4,DDO_154,55,0,0.14,6.20,7.89,50.51,6.2971,6.4400,6.2971,6.4400,0.0000
5,DDO_168,33,0,0.12,3.33,3.99,47.26,11.6999,11.9655,11.6999,11.9655,0.0000
6,DDO_210,8,0,0.04,2.68,0.31,12.02,35.6688,36.4784,35.6688,36.4784,0.0000
7,DDO_216,13,0,0.09,3.23,1.12,18.91,16.0664,16.4311,16.0664,16.4311,0.0000
8,DDO_43,9,0,0.47,8.56,4.19,33.22,7.2442,7.4086,7.2442,7.4086,0.0000
9,DDO_46,23,0,0.15,14.63,3.54,40.92,10.7086,10.9517,10.7086,10.9517,0.0000


## Result
Record in CORRECTIONS.md:
- the SHA-256 of each input
- the Data/Model counts from section 2
- the match count from section 2A
- the maximum difference from section 4

Per-galaxy values only. No population statistic is reported (FAIR2 paper, limitations 2, 3 and 10).